In [1]:
import joblib
import json
import pandas as pd

In [2]:
model = joblib.load("../models/artifacts/ridge_model.pkl")

In [3]:
with open ('../models/artifacts/feature_columns.json') as f:
    feature_columns = json.load(f)

# Extract Feature Importance

In [4]:
coefs = model.coef_
features = feature_columns

In [5]:
importance_df = pd.DataFrame({\
    'feature': features,
    'coefficient': coefs
})
importance_df['abs_coef'] = importance_df['coefficient'].abs()
importance_df = importance_df.sort_values('abs_coef', ascending=False)
importance_df

,feature,coefficient,abs_coef
14,district_大安區,103970.953424,103970.953424
10,district_北投區,-71645.969799,71645.969799
17,district_萬華區,-57614.474688,57614.474688
15,district_文山區,-56575.977002,56575.977002
11,district_南港區,-42129.194596,42129.194596
19,building_type_華廈(10層含以下有電梯),-35910.279616,35910.279616
8,district_信義區,28853.561592,28853.561592
5,log_area,-28759.384918,28759.384918
7,district_中正區,22821.336431,22821.336431
12,district_士林區,-22342.191144,22342.191144


# Group Features

In [7]:
def categorize_feature(feature):
    if feature.startswith('district_'):
        return 'location'
    elif feature.startswith('building_type_'):
        return 'building_type'
    elif feature in ['log_area', 'area_ratio']:
        return 'size'
    elif feature in ['total_floors']:
        return 'structure'
    elif feature in ['building_age']:
        return 'age'
    elif feature in ['parking_area', 'has_parking']:
        return 'parking'
    elif feature in ['time_index']:
        return 'time'
    else:
        return 'other'

In [8]:
importance_df['category'] = importance_df['feature'].apply(categorize_feature)

In [9]:
importance_df.groupby('category')['abs_coef'].mean().sort_values(ascending=False)

category
location         39580.517430
building_type    26635.285978
size             25263.883848
time              6278.229878
age               3140.927128
parking           3010.637328
structure          505.023976
Name: abs_coef, dtype: float64

In [10]:
importance_df.groupby('category')['abs_coef'].sum().sort_values(ascending=False)

category
location         435385.691734
building_type     53270.571956
size              50527.767695
time               6278.229878
parking            6021.274657
age                3140.927128
structure           505.023976
Name: abs_coef, dtype: float64

# Convert Coefficient to Meaning

### Key Insights
- Location dominates pricing
    - Large variation across districts
    - Premium areas (大安區, 信義區) significantly outperform others
- Larger properties have lower price per sqm
    - Indicates bulk pricing effect
    - Smaller units command higher per-unit value
- Building type matters
    - High-rise buildings outperform lower-density types
- Building age reduces value
    - Clear depreciation effect over time
- Structural features have limited influence
    - Floor-related variables are less impactful than expected
